<div align="center">

# COMSATS UNIVERSITY ISLAMABAD
### Department of Computer Science / Artificial Intelligence

## NATURAL LANGUAGE PROCESSING
### LAB 04  |  Text Preprocessing Techniques

</div>

| **Student Information** | **Details** |
|:---|:---|
| **Name** | Ayesha Mehmood |
| **Registration No.** | FA23-BAI-010 |
| **Instructor** | Sir Zaeem Akbar Khan |
| **University** | COMSATS University Islamabad |
| **Lab Number** | Lab 04 |

---

**Instructions:** Run each code cell from top to bottom. Sections follow the official lab task sheet. Example inputs are for demonstration, and can be changed as needed.

## Lab Tasks


### Setup and dependencies

The notebook uses NLTK, spaCy, matplotlib, pandas and scikit-learn. Some tasks need NLTK corpora and the trained `en_core_web_sm` model. Internet access is needed for the first download. Task 4 also offers optional Google Translate access; the fallback clearly identifies when online translation is unavailable.

In [ ]:
# Install once if needed:
# %pip install nltk spacy pandas matplotlib scikit-learn deep-translator cmudict
# !python -m spacy download en_core_web_sm
import re
import time
from collections import Counter
import nltk
import spacy
import pandas as pd
import matplotlib.pyplot as plt
from nltk.stem import PorterStemmer, SnowballStemmer
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score
# Get English stopwords (the first run may need internet).
try:
    from nltk.corpus import stopwords
    ENGLISH_STOP = set(stopwords.words('english'))
except LookupError:
    nltk.download('stopwords', quiet=True)
    try:
        ENGLISH_STOP = set(stopwords.words('english'))
    except LookupError:
        # Offline fallback only. Install NLTK stopwords for the exact assigned dataset.
        from spacy.lang.en.stop_words import STOP_WORDS
        ENGLISH_STOP = set(STOP_WORDS)
        print('Offline fallback: spaCy stopwords. Download NLTK stopwords for Task 3.')
porter = PorterStemmer()
snowball = SnowballStemmer('english')
try:
    nlp = spacy.load('en_core_web_sm')
    MODEL_READY = True
except OSError:
    nlp = spacy.blank('en')
    MODEL_READY = False
    print('Note: en_core_web_sm is not installed. Run the download command above for POS and lemmatization tasks.')
print('Setup complete.')

---
## Task 1: Lowercasing Basics and Its Drawbacks

Compare vocabulary sizes and consider loss of meaning after lowercasing.


### 1 to 3. Corpus, vocabulary comparison, and ambiguous words

In [ ]:
corpus = [
    "Apple makes new computers.", "I ate an apple today.",
    "APPLE announced a new product.", "The US has many universities.",
    "Please call us tomorrow.", "NASA studies space.",
    "My friend studies in Islamabad.", "The May meeting was cancelled.",
    "We may travel soon.", "Python is a popular language."
]
words_before = set(re.findall(r"[A-Za-z]+", " ".join(corpus)))
words_after = set(word.lower() for word in words_before)
print("Vocabulary before:", len(words_before))
print("Vocabulary after:", len(words_after))
print("Examples of lost distinctions:")
for pair in [("Apple", "apple"), ("US", "us"), ("May", "may")]:
    print(pair, "become:", pair[0].lower())

### 4. Observation

Lowercasing is often helpful for simple search, but it removes useful differences. **NER** may confuse a name with an ordinary word, **sentiment analysis** may lose capitalization emphasis, and **search** may mix acronyms, brands and common words.

---
## Task 2: Noise Removal Pipeline

Clean 15 noisy reviews, compare cleaning order, and identify overcleaning.


### 1 to 3. Cleaning functions and order comparison

In [ ]:
reviews = [
    "Amazing phone! 😊 https://shop.com", "@sara Great work!!!", "#NLP is fun 😊",
    "<b>LOVE</b> this course", "  Very   helpful!   ", "C++ is a good language",
    "This costs $100 only", "@uni #AI tools are cool", "Bad app 😠 <br> slow!",
    "Read more at www.example.org", "#MachineLearning rocks", "     Perfect service! ",
    "Email support@test.com", "I won't buy again!", "New update 10/10! 🚀"
]

def remove_urls(text): return re.sub(r'https?://\S+|www\.\S+', '', text)
def remove_html(text): return re.sub(r'<[^>]+>', '', text)
def remove_mentions_hashtags(text): return re.sub(r'[@#](\w+)', r'\1', text)
def remove_punctuation(text): return re.sub(r'[^\w\s]', ' ', text)
def remove_extra_spaces(text): return re.sub(r'\s+', ' ', text).strip()

def clean_text(text):
    text = remove_html(remove_urls(text))
    text = remove_mentions_hashtags(text)
    text = remove_punctuation(text)
    return remove_extra_spaces(text)

for old, new in [(x, clean_text(x)) for x in reviews[:5]]:
    print('BEFORE:', old, '\nAFTER: ', new, '\n')

example = '<b>Find NLP at https://example.org today!</b>'
first = clean_text(example)
second = remove_extra_spaces(remove_html(remove_urls(remove_punctuation(example))))
print('Normal order:', first)
print('Changed order:', second)
print('Does order change output?', first != second)

### 4. Five before and after examples with possible overcleaning

In [ ]:
from IPython.display import display
examples = ["C++ is great", "Price is $100", "#AI is cool", "@sara said hello", "Great 😊 quality!"]
display(pd.DataFrame({'Before': examples, 'After': [clean_text(x) for x in examples]}))
print('Overcleaning: removing punctuation changes C++ to C and $100 to 100; emojis are also lost.')

---
## Task 3: Stop Word Removal and Its Effect on Vocabulary

Stopword analysis and frequency graphs on sample reviews.


### 1 and 2. Stopword list and vocabulary statistics

In [ ]:
print('First 20 stopwords:', sorted(ENGLISH_STOP)[:20])
print('Total stopwords:', len(ENGLISH_STOP))
all_tokens = [w for text in reviews for w in re.findall(r"[a-z]+", clean_text(text.lower()))]
filtered_tokens = [w for w in all_tokens if w not in ENGLISH_STOP]
print('Before: tokens =', len(all_tokens), 'vocabulary =', len(set(all_tokens)),
      'average sentence length =', round(len(all_tokens) / len(reviews), 2))
print('After: tokens =', len(filtered_tokens), 'vocabulary =', len(set(filtered_tokens)),
      'average sentence length =', round(len(filtered_tokens) / len(reviews), 2))

### 3. Top 15 frequency charts

In [ ]:
for title, values in [('Before stopword removal', all_tokens), ('After stopword removal', filtered_tokens)]:
    common = Counter(values).most_common(15)
    plt.figure(figsize=(9, 3))
    plt.bar([w for w, n in common], [n for w, n in common])
    plt.title(title)
    plt.xticks(rotation=45, ha='right')
    plt.tight_layout()
    plt.show()

### 4. Custom stopwords and reasons

In [ ]:
# Domain words often repeated but not informative for this sample topic.
new_domain_words = {'course', 'app', 'phone', 'service', 'update', 'tools', 'work', 'read', 'buy', 'good'}
custom_stop = ENGLISH_STOP | new_domain_words
for word in sorted(new_domain_words):
    print(word, ': common domain/topic word; may dominate counts without adding much meaning')
print('Custom list size:', len(custom_stop))
print('Tokens left:', len([w for w in all_tokens if w not in custom_stop]))
# These 10 choices are task-specific; do not remove them for every NLP application.

---
## Task 4: Risks of Stop Word Removal (Sentiment, MT, Chatbots)

Demonstrate why blindly removing function words can change meaning.


### 1. Five sentiment sentences with negation

In [ ]:
sentiment = ["This is not good.", "I am never happy.", "There is no problem.",
             "I do not like this.", "The movie is not terrible."]
def without_stop(text, stop_list):
    return ' '.join(w for w in text.lower().split() if re.sub(r'[^a-z]', '', w) not in stop_list)
for s in sentiment:
    print(s, '=>', without_stop(s, ENGLISH_STOP))
print('Observation: deleting not, no or never can reverse or remove a sentiment meaning.')

### 2. English to Urdu comparison (online translator when available)

In [ ]:
translation_examples = [
    'I am not going to school.',
    'She does not like tea.',
    'We are waiting for the bus.'
]
try:
    from deep_translator import GoogleTranslator
    translator = GoogleTranslator(source='en', target='ur')
    for original in translation_examples:
        reduced = without_stop(original, ENGLISH_STOP)
        print('Original:', original, '| Urdu:', translator.translate(original))
        print('Reduced: ', reduced, '| Urdu:', translator.translate(reduced))
except Exception as e:
    print('Live translator unavailable. Install deep-translator and enable internet to run this part.')
    # The following are illustrative human-written translations, NOT a live translator result.
    urdu_original = ['میں اسکول نہیں جا رہا ہوں۔', 'اسے چائے پسند نہیں ہے۔', 'ہم بس کا انتظار کر رہے ہیں۔']
    urdu_reduced = ['اسکول جا رہا ہوں۔', 'چائے پسند ہے۔', 'بس کا انتظار ہے۔']
    for english, urdu_a, urdu_b in zip(translation_examples, urdu_original, urdu_reduced):
        print('Original:', english, '| Example Urdu:', urdu_a)
        print('Reduced: ', without_stop(english, ENGLISH_STOP), '| Example Urdu:', urdu_b)
    print('These offline Urdu phrases are only examples; verify them using an online translator.')
    print('Observation: removing not, does, am, are and for can distort grammatical meaning.')

### 3 and 4. Chatbot queries and safe stopword strategy

In [ ]:
queries = [
    "How can I not be charged?", "What is the status of my order?",
    "Can I cancel without paying?", "Why did I not get a refund?",
    "Is my account not active?"
]
for q in queries:
    print(q, '=>', without_stop(q, ENGLISH_STOP))
keep_negation = {'not', 'no', 'never', 'without', "don't", "isn't"}
safe_stop = ENGLISH_STOP - keep_negation
print('\nSafe results for original sentiment examples:')
for s in sentiment:
    print(s, '=>', without_stop(s, safe_stop))

---
## Task 5: Hands On with the Porter Stemmer

Study Porter and Snowball stemming on at least 30 words.


### 1 and 2. Stem words and record approximate correctness

In [ ]:
words = ['running','runs','runner','studies','studying','happily','happiness',
         'generalization','generally','better','was','playing','played','plays',
         'university','universe','connected','connection','connecting','cats',
         'cities','studied','quickly','activation','active','computers',
         'computation','relational','organization','beautifully','easily','fairness']
# Heuristic teaching labels. A stem is not always a proper dictionary word.
problem_examples = {'univers', 'wa', 'better', 'happili', 'gener'}
rows = [{'Word': w, 'Porter stem': porter.stem(w),
         'Correct/Incorrect (illustrative)': 'Review' if porter.stem(w) in problem_examples else 'Plausible'}
        for w in words]
display(pd.DataFrame(rows))

### 3 and 4. Stemming errors and Snowball comparison

In [ ]:
print('Overstemming examples:')
for a, z in [('university','universe'), ('general','generous'), ('organize','organization')]:
    print(a, z, '=>', porter.stem(a), porter.stem(z))
print('Understemming examples (related words may keep different roots):')
for a, z in [('analysis','analyze'), ('good','better'), ('decide','decision')]:
    print(a, z, '=>', porter.stem(a), porter.stem(z))
print('Porter vs Snowball differences:')
for word in words:
    if porter.stem(word) != snowball.stem(word):
        print(word, porter.stem(word), snowball.stem(word))
# Inspect the displayed stems: shared stems can be misleading and related words can stay different.

---
## Task 6: Lemmatization (spaCy) vs Stemming

Compare word stems, POS tags, and lemmas for a 100 word paragraph.


### 1 to 3. Lemmas, parts of speech, and stem comparisons

In [ ]:
paragraph = """Students are learning natural language processing because computers need to understand
human messages. Researchers study how different words can express similar ideas and how
sentences are arranged. A student was reading an article and noticed that better results
came from cleaning the data carefully. Several friends were writing programs that compare
words, correct spelling mistakes and organize the documents into useful categories.
The university project includes short reviews about products, studies of sentiment,
and experiments with machine learning. Some models are faster than others but may
not understand context well. The team plans to examine these differences and write
clear reports about the results. They are working together, testing examples, and
explaining what was learned from the experiments."""
print('Word count:', len(paragraph.split()))
if MODEL_READY:
    document = nlp(paragraph)
    result = pd.DataFrame([{'Token': t.text, 'POS': t.pos_, 'Stem': porter.stem(t.text.lower()),
                            'Lemma': t.lemma_} for t in document if t.is_alpha])
    display(result.head(30))
    for word in ['better','was','were','studies']:
        matches = result[result.Token.str.lower() == word]
        if not matches.empty: print(word, '=>', matches[['POS', 'Stem', 'Lemma']].values.tolist())
    print('POS helps lemmatization distinguish word roles and select the dictionary root.')
else:
    print('Trained spaCy model required for meaningful POS and lemmas. Download en_core_web_sm and rerun.')

### 4. Measure processing time

In [ ]:
words_sample = paragraph.lower().split()
t0 = time.perf_counter()
stems = [porter.stem(w) for w in words_sample]
stem_time = time.perf_counter() - t0
print('Porter time:', round(stem_time, 6), 'seconds')
if MODEL_READY:
    t0 = time.perf_counter()
    lemma_list = [t.lemma_ for t in nlp(paragraph) if t.is_alpha]
    lemma_time = time.perf_counter() - t0
    print('spaCy time:', round(lemma_time, 6), 'seconds')
    print('Choose stemming for speed, and lemmatization for context-sensitive dictionary forms.')
else:
    print('spaCy timing skipped because trained model is unavailable.')

---
## Task 7: Preprocessing Pipeline

Run four NLP preprocessing options on 60 labeled reviews and compare classification accuracy.


### 1 and 2. Preprocessing function and dataset of 60 documents

In [ ]:
positive = ['great','excellent','amazing','useful','wonderful','reliable','smooth','fast','helpful','pleasant']
negative = ['bad','poor','awful','broken','terrible','unreliable','slow','buggy','useless','unpleasant']
# 30 unique examples per class: 10 adjectives and 3 text templates each.
reviews_data = []
for label, adjectives in [(1, positive), (0, negative)]:
    for adj in adjectives:
        reviews_data.extend([(f'The service is {adj} and I would talk about it again.', label),
                             (f'I found this product {adj} after using it for a week.', label),
                             (f'My overall experience with this application was {adj}.', label)])
print('Number of documents:', len(reviews_data))

def preprocess(text, method='lower'):
    text = re.sub(r'[^a-z ]', ' ', text.lower())
    tokens = text.split()
    if method in ('stop', 'lemma'):
        tokens = [w for w in tokens if w not in ENGLISH_STOP]
    if method == 'stem':
        tokens = [porter.stem(w) for w in tokens]
    if method == 'lemma' and MODEL_READY:
        tokens = [t.lemma_ for t in nlp(' '.join(tokens)) if t.is_alpha]
    return ' '.join(tokens)

methods = ['lower', 'stop', 'stem', 'lemma']
# Use the same training/test split for a fair comparison.
texts = [x for x, label in reviews_data]
labels = [label for x, label in reviews_data]
train_ids, test_ids = train_test_split(list(range(len(texts))), test_size=0.3,
                                     random_state=42, stratify=labels)

### 3 and 4. Statistics, model training, and comparison

In [ ]:
if not MODEL_READY:
    print('Lemmatization result cannot be compared without en_core_web_sm.')
comparison = []
for method in methods:
    if method == 'lemma' and not MODEL_READY: continue
    clean_docs = [preprocess(x, method) for x in texts]
    vectorizer = TfidfVectorizer()
    X_train = vectorizer.fit_transform([clean_docs[i] for i in train_ids])
    X_test = vectorizer.transform([clean_docs[i] for i in test_ids])
    classifier = LogisticRegression(max_iter=500)
    classifier.fit(X_train, [labels[i] for i in train_ids])
    accuracy = accuracy_score([labels[i] for i in test_ids], classifier.predict(X_test))
    vocabulary = {w for document in clean_docs for w in document.split()}
    comparison.append({'Method': method, 'Vocabulary size': len(vocabulary),
                       'Avg tokens/document': round(sum(len(x.split()) for x in clean_docs) / len(clean_docs), 2),
                       'Accuracy': round(accuracy, 3)})
results = pd.DataFrame(comparison)
display(results)
print('Best configuration for this demo:', results.loc[results.Accuracy.idxmax(), 'Method'])

### Analysis (Task 7)

This experiment compares the same 60 short examples using the same training and testing split. Lowercasing keeps more word forms, while removing stopwords may reduce vocabulary and sentence length. Stemming combines some related forms; lemmatization also uses grammatical information when a trained model is available. The accuracy table should be used to decide the best configuration rather than assuming one method is always superior. Because this is a small synthetic dataset, its accuracy is only a teaching result and not a reliable real world benchmark.

---
## Task 8: Minimum Edit Distance (Levenshtein)

Implement edit distance recursively and with dynamic programming, including a backtrace.


### 1 and 2. Recursive and iterative Levenshtein distance

In [ ]:
def distance_recursive(a, b):
    if not a: return len(b)
    if not b: return len(a)
    cost = 0 if a[0] == b[0] else 1
    return min(1 + distance_recursive(a[1:], b),
               1 + distance_recursive(a, b[1:]),
               cost + distance_recursive(a[1:], b[1:]))

def distance_table(a, b, substitute_cost=1):
    dp = [[0] * (len(b) + 1) for _ in range(len(a) + 1)]
    for i in range(len(a) + 1): dp[i][0] = i
    for j in range(len(b) + 1): dp[0][j] = j
    for i in range(1, len(a) + 1):
        for j in range(1, len(b) + 1):
            cost = 0 if a[i - 1] == b[j - 1] else substitute_cost
            dp[i][j] = min(dp[i-1][j] + 1, dp[i][j-1] + 1, dp[i-1][j-1] + cost)
    return dp

print('Recursive distance cat -> cut:', distance_recursive('cat', 'cut'))
print('Dynamic programming distance cat -> cut:', distance_table('cat', 'cut')[-1][-1])
# Plain recursion becomes very slow for long words because it repeats subproblems.

### 3. DP tables for all three required pairs

In [ ]:
pairs = [('intention','execution'), ('kitten','sitting'), ('sunday','saturday')]
for left, right in pairs:
    table = distance_table(left, right)
    print(f'\n{left} -> {right}; distance = {table[-1][-1]}')
    display(pd.DataFrame(table, index=['∅'] + list(left), columns=['∅'] + list(right)))
# These tables show the same cell-by-cell values needed for a handwritten solution.

### 4. Backtrace operations and substitution cost 2

In [ ]:
def backtrace(a, b, sub_cost=1):
    dp = distance_table(a, b, sub_cost)
    i, j = len(a), len(b)
    operations = []
    while i or j:
        if i and j and a[i-1] == b[j-1] and dp[i][j] == dp[i-1][j-1]:
            i -= 1; j -= 1  # matching character
        elif i and j and dp[i][j] == dp[i-1][j-1] + sub_cost:
            operations.append(f"Substitute {a[i-1]} with {b[j-1]}")
            i -= 1; j -= 1
        elif i and dp[i][j] == dp[i-1][j] + 1:
            operations.append(f"Delete {a[i-1]}")
            i -= 1
        else:
            operations.append(f"Insert {b[j-1]}")
            j -= 1
    return list(reversed(operations))

for left, right in pairs:
    for cost in [1, 2]:
        print(left, '->', right, '| substitution cost:', cost,
              '| distance:', distance_table(left, right, cost)[-1][-1],
              '| operations:', backtrace(left, right, cost))

---
## Task 9: Spelling Correction with Edit Distance

Use a 5,000 plus word vocabulary and edit distance to suggest corrections.


### 1. Vocabulary and word frequency data

In [ ]:
# TextBlob ships a frequency dictionary with over 29,000 entries.
from textblob.en import spelling
freq_words = {w: int(count) for w, count in spelling.items() if w.isalpha()}
print('Vocabulary size:', len(freq_words))
assert len(freq_words) >= 5000

### 2. Suggest the k nearest dictionary words

In [ ]:
def edit_distance(a, b):
    # A memory-efficient dynamic programming implementation.
    previous = list(range(len(b) + 1))
    for i, char_a in enumerate(a, 1):
        current = [i]
        for j, char_b in enumerate(b, 1):
            current.append(min(current[-1] + 1, previous[j] + 1,
                               previous[j - 1] + (char_a != char_b)))
        previous = current
    return previous[-1]

def suggest(word, k=5, use_frequency=False):
    word = word.lower()
    # First-letter and length filters make this small lab experiment quick.
    candidates = [w for w in freq_words if abs(len(w) - len(word)) <= 2
                  and w[0] == word[0] and w != word]
    scored = [(edit_distance(word, w), w) for w in candidates]
    if use_frequency:
        scored.sort(key=lambda pair: (pair[0], -freq_words[pair[1]], pair[1]))
    else:
        scored.sort(key=lambda pair: (pair[0], pair[1]))
    return [w for d, w in scored[:k]]

print("Suggestions for 'recieve':", suggest('recieve'))

### 3 and 4. Test 15 mistakes, compute Top 1 accuracy and break ties by frequency

In [ ]:
mistakes = {
    'recieve':'receive', 'definately':'definitely', 'acommodate':'accommodate',
    'langauge':'language', 'speling':'spelling', 'teh':'the', 'acheive':'achieve',
    'enviroment':'environment', 'becuase':'because', 'goverment':'government',
    'adress':'address', 'aplication':'application', 'freind':'friend',
    'knolwedge':'knowledge', 'succesful':'successful'
}
rows = []
for wrong, correct in mistakes.items():
    basic = suggest(wrong, 1)
    ranked = suggest(wrong, 1, use_frequency=True)
    rows.append({'Mistake': wrong, 'Expected': correct,
                 'Edit distance only': basic[0] if basic else '',
                 'Distance + frequency': ranked[0] if ranked else ''})
results_spelling = pd.DataFrame(rows)
display(results_spelling)
accuracy_basic = (results_spelling['Edit distance only'] == results_spelling.Expected).mean()
accuracy_ranked = (results_spelling['Distance + frequency'] == results_spelling.Expected).mean()
print(f'Top 1 accuracy with distance only: {accuracy_basic:.1%}')
print(f'Top 1 accuracy with frequency ties: {accuracy_ranked:.1%}')
print('Real word errors such as their/there cannot be detected by dictionary spelling alone.')
print('Limitation: the fast candidate filter may miss words starting with a different letter.')